# QuFoods Exploration — Data Profiling, Quality Issues, and Imputation Validation

Owner: Siyan (Adetu Siyanbola) — Exploration workstream, Implementation Plan tasks #8–22.

This notebook builds directly on `qufoods_starter_notebook.ipynb` — same `list_recent_keys` / `load_batches` ingestion approach, same `clean_order_items` typo-fix mechanics — but moves the reusable logic into the `exploration` package (`src/exploration/`) so it can be imported by Emmanuel's Lambda, tested with `pytest`, and run unattended. This notebook is the **narrative layer** on top of that package: it calls the package functions and explains *why* each decision was made, which is what actually needs to go in the data quality log and the team deck.

**Run modes:** by default this runs against the bundled sample batch in `data/sample_batches/` (zero AWS dependency — works the moment you clone the repo). Flip `USE_S3 = True` below once IAM access to `qufoods-raw` is confirmed, and every cell below runs unchanged against the live feed.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')  # exploration package lives in exploration/src/exploration

import pandas as pd
import matplotlib.pyplot as plt

from exploration.pipeline import run, split_record_types
from exploration.ingest import pull_batches
from exploration.reference import load_branches, load_menu_items, FLAGGED_BRANCHES
from exploration.profiling import field_profile, branch_quality_profile, compare_flagged_vs_other, duplicate_summary
from exploration.cleaning import clean_order_items, TYPO_MATCH_CUTOFF
from exploration.imputation import validate_formula, formula_total

%matplotlib inline
pd.set_option('display.max_columns', None)

In [ ]:
# Flip to True once IAM access (s3:ListBucket + s3:GetObject on qufoods-raw) is confirmed.
# Every cell below is unchanged either way -- this is the only switch.
USE_S3 = False

AWS_PROFILE = 'qufoods'   # set to None to use your default AWS credential chain
LOOKBACK_MINUTES = 1440   # wide window for exploration; production Lambda uses 5
SAMPLE_DIR = '../data/sample_batches'

## Part 1 — Pull raw batches

Same approach as the starter notebook's `list_recent_keys` + `load_batches`, wrapped behind `pull_batches()` so the local/live switch is one argument instead of two different code paths to maintain. The bundled sample batch (`BATCH-96bd24c2...`) is the exact one pulled from the public URL given in the lab brief.

In [ ]:
ingest_result = pull_batches(use_s3=USE_S3, minutes=LOOKBACK_MINUTES, profile=AWS_PROFILE, sample_dir=SAMPLE_DIR)
raw_df = ingest_result.raw_df

print(f"{len(ingest_result.source_keys)} batch file(s) loaded")
print(f"{len(raw_df)} total records")
raw_df.sample(1).to_dict(orient='records')[0]

## Part 2 — Split sales vs. expenses

~95% sales, ~5% expense per the brief — worth confirming that holds, since a wildly different split on a real pull would be a signal something upstream changed.

In [ ]:
sales_df, expense_df = split_record_types(raw_df)

print('sales:  ', sales_df.shape, f"({len(sales_df)/len(raw_df):.0%} of batch)")
print('expense:', expense_df.shape, f"({len(expense_df)/len(raw_df):.0%} of batch)")

## Part 3 — Field profiling (Task #10)

Null rate, dtype, distinct count, and (for numerics) min/max per column. Run separately on sales and expense since they're different schemas after the split.

In [ ]:
sales_profile = field_profile(sales_df)
sales_profile

In [ ]:
expense_profile = field_profile(expense_df)
expense_profile

**Reading the null rates:** not every null is a data quality problem. `membership_id` being null most of the time is a *real value* — most customers are walk-ins, not members. `approved_by` being null on expenses means the expense hasn't been approved yet — also real. The nulls that actually matter are `total_amount` (the brief's named issue) and `branch_name`/`branch_manager` (a digitisation artifact, fixed in Part 5).

## Part 4 — Branch-level quality deep dive (Task #11)

The brief specifically calls out **BR-06, BR-07** (Ogun State) and **BR-15, BR-16, BR-17** (south-east) as having known data discipline issues. Confirming this with evidence rather than taking it on faith — and checking whether any *other* branch is quietly just as bad, since the brief explicitly says these aren't the only ones.

In [ ]:
branch_quality = branch_quality_profile(sales_df, expense_df)
branch_quality.sort_values('total_amount_null_rate', ascending=False)

In [ ]:
compare_flagged_vs_other(branch_quality)

**Note on sample size:** the bundled sample batch is a single ~26-record file — too small for this flagged-vs-other comparison to be statistically meaningful on its own. Once pulling across multiple dates from the live bucket (`USE_S3 = True`, wider `LOOKBACK_MINUTES`), re-run this cell and look for the gap between flagged and non-flagged branches to widen. If it doesn't, that's itself a finding worth documenting — it would mean the brief's branch-specific framing doesn't fully hold and the real quality issues are more evenly distributed than described.

## Part 5 — Repair branch fields from the reference registry

`branch_name` / `branch_manager` nulls are a digitisation artifact, not a real gap — `branch_id` is never null and the registry is always internally consistent. Recovered via lookup, not dropped. (`exploration.branch_repair.repair_branch_fields` — same logic as the starter notebook's Part 5, factored into the package.)

In [ ]:
from exploration.branch_repair import repair_branch_fields

before = sales_df['branch_name'].isna().mean()
sales_df = repair_branch_fields(sales_df)
after = sales_df['branch_name'].isna().mean()
print(f"branch_name null rate: {before:.1%} -> {after:.1%}")

## Part 6 — Lock the region mapping (Task #12) and confirm the menu list (Task #13)

Region mapping is a business decision (which states count as "West" vs "Other"), locked once in `exploration.reference.STATE_TO_REGION` so Bukolami's schema and Daniel's regional reports use the same definition. The 13-item menu list comes straight from `reference_menu_items.csv`.

In [ ]:
branches = load_branches()
branches[['branch_id', 'branch_name', 'state', 'region']]

In [ ]:
branches['region'].value_counts()

In [ ]:
menu_items = load_menu_items()
print(f"{len(menu_items)} canonical menu items:")
menu_items

## Part 7 — Misspelling correction (Tasks #14–15)

`exploration.cleaning.clean_order_items` mirrors the starter notebook's mechanics but swaps `difflib` for `rapidfuzz` and changes the cutoff from the implementation plan's suggested 85% down to **75%** — backed by evidence, not a guess. See the module docstring in `src/exploration/cleaning.py` for the full reasoning; short summary below.

In [ ]:
from exploration.cleaning import score_candidates, correct_item_name

# The real typos found in this sample batch:
real_typos = ['ocke', 'ckoe', 'chickne wings', 'ofada irce', 'chicekn wings']
for typo in real_typos:
    scores = score_candidates(typo, menu_items)
    print(f"{typo!r:20s} -> best match {scores[0]}")

**Why 75 and not the plan's suggested 85:** at 85, `"ocke"` and `"ckoe"` — both real typos of `"coke"` — score only 75, so they'd silently fail to get corrected. Short item names lose more similarity percentage per swapped letter than long ones (a 4-letter swap on `coke` costs far more than a 1-letter swap on `chicken wings`). Checking the other direction — genuinely off-menu Nigerian dishes (`suya`, `amala`, `moi moi`, `akara`, `beans`) — the highest any of them scores against this menu is 50. That leaves a wide, evidence-backed gap between 50 and 75 with nothing observed in between, so 75 is the cutoff actually used in `TYPO_MATCH_CUTOFF`.

In [ ]:
off_menu_dishes = ['spaghetti carbonara', 'pizza', 'suya', 'amala', 'moi moi', 'akara', 'beans']
for dish in off_menu_dishes:
    scores = score_candidates(dish, menu_items)
    print(f"{dish!r:20s} -> best match {scores[0]}")

In [ ]:
cleaned = sales_df['order_items'].apply(lambda v: clean_order_items(v, menu_items, TYPO_MATCH_CUTOFF))
sales_df['order_items_clean'] = cleaned.apply(lambda t: t[0])
sales_df['order_items_typo_fixed'] = cleaned.apply(lambda t: t[1])

n_fixed = sales_df['order_items_typo_fixed'].sum()
print(f"typo-fixed {n_fixed} of {len(sales_df)} orders")
sales_df.loc[sales_df['order_items_typo_fixed'], ['order_items', 'order_items_clean']]

## Part 8 — Validate the imputation formula (Task #16)

**This is the single most important finding in the whole exploration phase.** The brief and the architecture diagram describe the relationship as `order_subtotal - discount_applied = total_amount` (flat subtraction). Testing that against real records with all three fields present shows it's wrong — `discount_applied` is a **fraction (0–1)**, a discount *rate*, not a currency amount. The correct relationship is **multiplicative**: `total_amount = order_subtotal * (1 - discount_applied)`.

In [ ]:
# Negative control -- proves the brief's literal wording is wrong
sample_check = sales_df.dropna(subset=['order_subtotal', 'discount_applied', 'total_amount'])
sample_check = sample_check[sample_check['discount_applied'] > 0].head(5)

sample_check['naive_subtraction'] = sample_check['order_subtotal'] - sample_check['discount_applied']
sample_check['correct_multiplicative'] = formula_total(sample_check['order_subtotal'], sample_check['discount_applied'])

sample_check[['transaction_id', 'order_subtotal', 'discount_applied', 'total_amount', 'naive_subtraction', 'correct_multiplicative']]

In [ ]:
result = validate_formula(sales_df)
print(result)
print(f"match rate: {result.match_rate:.1%} on {result.n_checked} complete rows")
result.example_mismatches

## Part 9 — Decide the imputation method (Task #17) and apply it (Tasks #18–19)

Given the formula validates near-100% on complete rows, the strategy is:

1. **Algebraic solve first** — whenever exactly one of `{order_subtotal, discount_applied, total_amount}` is missing, solve for it directly from the other two. No model needed, no training data needed, and it's *exact*, not an estimate.
2. **Regression fallback** — only for the rarer case where 2+ fields are missing at once and the algebraic solve has nothing to work with. Trained on features that don't depend on the missing fields themselves (`branch_id`, `order_channel`, `payment_method`, item count) so it stays usable on exactly the rows that need it.

In [ ]:
from exploration.imputation import apply_algebraic_imputation, build_regression_fallback, apply_regression_fallback, sanity_check_imputed

n_missing_before = sales_df['total_amount'].isna().sum()
sales_df = apply_algebraic_imputation(sales_df)
n_algebraic = (sales_df['imputation_method'] == 'algebraic').sum()
print(f"missing total_amount before: {n_missing_before}")
print(f"resolved algebraically: {n_algebraic}")

In [ ]:
model = build_regression_fallback(sales_df)
sales_df = apply_regression_fallback(sales_df, model)

n_still_missing = sales_df['total_amount'].isna().sum()
print(f"still missing after algebraic pass: {n_still_missing}")
print(f"regression model trained: {model is not None}")

**Note:** the bundled sample batch is too small to have any rows with 2+ missing fields at once, and too small (well under the 30-row `MIN_TRAINING_ROWS` threshold) to train a real regression model — so `model` will be `None` here, and the fallback path falls back further to a branch-level median. This is exactly the kind of thing that needs re-validating once pulling a real multi-day sample from the live bucket.

## Part 10 — Sanity check imputed values (Task #20)

**Note on a bug found during review:** the original version of this check compared `total_amount` against `order_subtotal` to catch implausible imputed values — but on rows where `order_subtotal` is *also* missing (the regression-fallback case), that comparison silently evaluates to `NaN`, which pandas treats as falsy. The check was passing every regression-fallback row by accident, regardless of how implausible the predicted value was. Fixed by branching explicitly: use the subtotal comparison when a subtotal exists, fall back to a branch-median-based heuristic (flag anything >3x or <1/3 of the branch's typical total) when it doesn't.

In [ ]:
sales_df = sanity_check_imputed(sales_df)
n_implausible = sales_df['imputation_flagged_implausible'].sum()
print(f"imputed rows flagged as implausible: {n_implausible}")
sales_df[sales_df['imputation_flagged_implausible']]

## Part 11 — Other data quality issues found (Task #21)

Beyond the two issues the brief names explicitly, here's what else turned up. Full detail and resolution reasoning lives in `DATA_QUALITY_LOG.md` at the repo root — this is the short version inline.

### order_subtotal range sanity

`order_subtotal` is the one money field the brief describes as "always present" and never imputed — so if it's ever negative, zero, or implausibly large, that's an upstream data problem, not something to impute around. Flagging anything non-positive outright, and anything above an IQR-based outlier fence (adapts to real branch spend patterns rather than a hardcoded NGN figure tuned to this one sample batch).

In [ ]:
from exploration.profiling import plausibility_checks

plausibility = plausibility_checks(sales_df)
print(f"subtotal anomalies: {plausibility['n_subtotal_anomalies']} (upper fence: NGN {plausibility['subtotal_upper_fence_used']:,.2f})")
plausibility['subtotal_anomalies']

### Empty order_items on a COMPLETED sale

A completed transaction with a populated `total_amount` but no items listed implies money changed hands for nothing — a real anomaly, not just "zero items sold." Checked separately from FAILED transactions, where an empty `order_items` is plausible (the order may have failed before items were attached).

In [ ]:
print(f"empty order_items on COMPLETED sales: {plausibility['n_empty_items_on_completed_sales']}")
plausibility['empty_items_on_completed_sales']

### Duplicate ID check

In [ ]:
dup_sales = duplicate_summary(sales_df, 'transaction_id')
dup_expense = duplicate_summary(expense_df, 'record_id')
print('sales duplicates:  ', dup_sales)
print('expense duplicates:', dup_expense)

In [ ]:
# transaction_status includes FAILED -- these still have a populated total_amount,
# meaning 'revenue by branch' naively summed would overstate real revenue unless filtered.
sales_df['transaction_status'].value_counts()

In [ ]:
# order_source vs order_channel sometimes disagree (e.g. order_channel=IN_STORE but
# order_source=ONLINE) -- worth flagging to Bukolami as a schema question: which field
# is authoritative for channel-based reporting?
pd.crosstab(sales_df['order_channel'], sales_df['order_source'])

In [ ]:
# customer_departure_time before customer_arrival_time on at least one record
# (timestamp data entry issue, not a calculation bug) -- check for it explicitly:
ts = sales_df[['transaction_id', 'customer_arrival_time', 'customer_departure_time']].copy()
ts['arrival'] = pd.to_datetime(ts['customer_arrival_time'])
ts['departure'] = pd.to_datetime(ts['customer_departure_time'])
ts['negative_dwell'] = ts['departure'] < ts['arrival']
ts[ts['negative_dwell']]

## Part 12 — Quick analytics pass (sanity check, not the final report)

Same charts as the starter notebook's Part 8, run on the cleaned + imputed frame instead of the naive one — just to eyeball that nothing looks structurally broken before handing off.

In [ ]:
revenue_by_branch = sales_df.groupby('branch_id')['total_amount'].sum().sort_values(ascending=False)
revenue_by_branch.plot(kind='bar', figsize=(10, 4), title='Total revenue by branch (cleaned + imputed)')
plt.ylabel('NGN')
plt.show()

In [ ]:
item_counts = (
    sales_df['order_items_clean']
    .str.split(', ')
    .explode()
    .str.replace(r'\(x\d+\)$', '', regex=True)
    .value_counts()
)
item_counts.plot(kind='bar', figsize=(10, 4), title='Most ordered items (typo-corrected)')
plt.show()

## Part 13 — Package for handoff to Emmanuel (Task #22)

Everything above is also available as a single function call — `exploration.pipeline.run()` — so Emmanuel's Lambda cleaning module doesn't need to re-derive any of this logic; it imports from this package directly. See `HANDOFF.md` at the repo root for the full contract (function signatures, what each one expects/returns, and exactly which constants are safe to import rather than copy).

In [ ]:
from exploration.pipeline import run

result = run(use_s3=USE_S3, minutes=LOOKBACK_MINUTES, profile=AWS_PROFILE, sample_dir=SAMPLE_DIR)

print(f"raw records:              {result.raw_record_count}")
print(f"sales / expense:          {len(result.sales_df)} / {len(result.expense_df)}")
print(f"typo corrections:         {result.n_typo_corrections}")
print(f"algebraic imputations:    {result.n_algebraic_imputations}")
print(f"regression imputations:   {result.n_regression_imputations}")
print(f"flagged as implausible:   {result.n_implausible_imputations}")
print(f"formula match rate:       {result.formula_validation.match_rate:.1%}")

### Next steps for this section

1. Re-run this whole notebook with `USE_S3 = True` the moment IAM access to `qufoods-raw` is confirmed — pull several days' worth of files, not just the one sample batch, and re-check every conclusion above (especially the flagged-vs-other branch comparison and the regression fallback, both of which need more data than one batch provides to mean anything).
2. Update `DATA_QUALITY_LOG.md` with anything new that turns up on the larger pull.
3. Hand off `src/exploration/` to Emmanuel as-is — it's already an importable package (`pip install -e .`), so his Lambda cleaning module can `from exploration.cleaning import clean_order_items` and `from exploration.imputation import apply_algebraic_imputation` directly instead of re-implementing.